
# 05 — Análisis Ejecutivo de Resultados y Recomendaciones para Comité Directivo

**Proyecto:** RADAR Cibest — Selección Internacional de Mercados  
**Audiencia:** Comité Directivo / CFO / Alta Dirección  
**Propósito:** traducir los resultados de RADAR y del análisis de robustez en una narrativa ejecutiva de decisión: qué mercados deberían avanzar a profundización, por qué, con qué tesis de negocio, con qué robustez y con qué restricciones.

---

## 0. Tesis ejecutiva del notebook

- Se evaluó el atractivo relativo de mercados internacionales para Grupo Cibest, combinando fundamentos estructurales, proximidad con Colombia, momentum macroeconómico y tesis por línea de negocio.
- El atractivo se define como una combinación de potencial, riesgo, ejecutabilidad, fit con la tesis de negocio y robustez de la señal.
- RADAR entrega un sistema de priorización y diagnóstico: ranking, perfiles, drivers, restricciones y señales por línea de negocio.
- Monte Carlo no prueba éxito comercial; valida si la recomendación es estable ante incertidumbre razonable en pesos y supuestos del modelo.
- El Comité debe decidir qué países pasan a profundización cualitativa, qué líneas se evalúan en cada país y qué restricciones deben validarse primero.
- No debe inferirse que RADAR recomienda entrar a un mercado; **RADAR no decide dónde entrar; RADAR identifica dónde tiene más sentido profundizar.**



## 1. Qué significa atractivo para Cibest

**Atractivo no es tamaño de mercado. Atractivo es la combinación entre potencial, riesgo, ejecutabilidad, fit con la tesis de negocio y robustez de la señal.**

Para Grupo Cibest, el atractivo de un país se interpreta en cinco capas:

1. **Atractivo estructural:** fundamentos macroeconómicos, profundidad financiera, institucionalidad e infraestructura digital.
2. **Ejecutabilidad:** proximidad cultural, geográfica, lingüística, migratoria o comercial con Colombia.
3. **Momentum:** dinámica reciente de crecimiento y mejora relativa.
4. **Encaje por línea de negocio:** cada negocio tiene drivers distintos; un país puede ser atractivo para Pagos y Flujos y no serlo para CIB.
5. **Robustez:** la prioridad debe mantenerse razonablemente estable ante cambios plausibles en pesos, supuestos y escenarios.

La consecuencia estratégica es directa: **un ranking global sirve para ordenar el universo, pero la decisión real se toma por país, línea de negocio, driver, restricción y robustez.**



## 2. Resumen del proceso SIM y lugar de RADAR

El proceso de Selección Internacional de Mercados —SIM— no es un modelo de scoring. Es un embudo de decisión que reduce progresivamente el universo de mercados con criterios de creciente profundidad.

> **Advertencia conceptual:** “Un mercado que no supera un filtro del embudo no es un mercado descartado permanentemente; es un mercado que no es el candidato correcto para Cibest en el horizonte de tiempo de la estrategia actual y con las capacidades actuales.”

**RADAR corresponde a la Fase 3. Sus resultados deben activar análisis posterior, no cerrar la decisión.**


In [56]:

from __future__ import annotations

import json
import math
import re
import sys
import warnings
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, Markdown, display

warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# Visual style
# -----------------------------------------------------------------------------
CIBEST: Dict[str, str] = {
    "gray": "#2C2A28",
    "gray_light": "#CCCAC7",
    "yellow": "#FDD923",
    "gold": "#D6B302",
    "gold_light": "#FFF7D3",
    "gold_dark": "#8F7701",
    "gray_bg": "#F5F5F5",
    "gray_border": "#D0D0D0",
    "white": "#FFFFFF",
    "green": "#0BA682",
    "amber": "#FF7E41",
    "red": "#C62828",
    "blue": "#1B4D6E",
}

TIER_COLORS: Dict[str, str] = {
    "Alta": CIBEST["green"],
    "Media-alta": CIBEST["gold"],
    "Media": CIBEST["amber"],
    "Baja": CIBEST["red"],
}


def style_table(
    df: pd.DataFrame,
    gradient_cols: Optional[List[str]] = None,
    gradient_cmap: str = "YlGnBu",
    format_dict: Optional[Dict[str, str]] = None,
):
    """Aplica estilo ejecutivo Cibest a una tabla pandas."""
    styler = df.style.set_table_styles([
        {"selector": "th", "props": [
            ("background-color", CIBEST["gray"]),
            ("color", CIBEST["yellow"]),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("padding", "8px"),
            ("font-family", "Arial, sans-serif"),
        ]},
        {"selector": "td", "props": [
            ("padding", "6px 10px"),
            ("font-family", "Arial, sans-serif"),
            ("border-bottom", f"1px solid {CIBEST['gray_border']}"),
        ]},
    ])
    if gradient_cols:
        cols = [c for c in gradient_cols if c in df.columns]
        if cols:
            styler = styler.background_gradient(subset=cols, cmap=gradient_cmap)
    if format_dict:
        styler = styler.format(format_dict)
    return styler


def insight_box(title: str, text: str) -> None:
    """Muestra una caja ejecutiva de insight."""
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['gold']}; background-color:{CIBEST['gold_light']};
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def risk_box(title: str, text: str) -> None:
    """Muestra una caja ejecutiva de advertencia."""
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['red']}; background-color:#FDECEC;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def success_box(title: str, text: str) -> None:
    """Muestra una caja ejecutiva de confirmación."""
    display(HTML(f"""
    <div style="border-left:6px solid {CIBEST['green']}; background-color:#E8F7F3;
                padding:14px 18px; margin:12px 0; font-family:Arial, sans-serif; color:{CIBEST['gray']};">
        <b>{title}</b><br>{text}
    </div>
    """))


def warn_missing(name: str, action: str) -> None:
    """Estandariza advertencias por faltantes de insumos."""
    risk_box(
        f"Insumo no disponible: {name}",
        f"El notebook intentará reconstruirlo desde los archivos disponibles. Si no es posible, {action}."
    )


In [57]:

# -----------------------------------------------------------------------------
# SIM funnel visualization
# -----------------------------------------------------------------------------
sim_funnel = pd.DataFrame([
    {"fase": 1, "etapa": "Objetivo estratégico", "pregunta": "¿Para qué se internacionaliza Cibest?", "output": "Tesis y criterios de decisión", "siguiente": "Define universo, pesos y tesis por línea"},
    {"fase": 2, "etapa": "Screening primario", "pregunta": "¿Qué mercados son viables?", "output": "Mercados admisibles", "siguiente": "Evita analizar mercados inviables"},
    {"fase": 3, "etapa": "RADAR", "pregunta": "¿Dónde tiene sentido profundizar?", "output": "Ranking, perfiles, drivers, señales", "siguiente": "Activa profundización cualitativa"},
    {"fase": 4, "etapa": "Inteligencia cualitativa", "pregunta": "¿Qué explica la oportunidad y el riesgo?", "output": "Diagnóstico sectorial y competitivo", "siguiente": "Valida o ajusta la señal cuantitativa"},
    {"fase": 5, "etapa": "Capacidades Cibest", "pregunta": "¿Puede Cibest capturar la oportunidad?", "output": "Gap de capacidades", "siguiente": "Ajusta priorización"},
    {"fase": 6, "etapa": "Priorización ajustada", "pregunta": "¿Qué mercados quedan en shortlist?", "output": "Candidatos perfilados", "siguiente": "Define modo de entrada"},
    {"fase": 7, "etapa": "Modo de entrada", "pregunta": "¿Cómo entrar?", "output": "Build / buy / partner / digital", "siguiente": "Prepara due diligence"},
    {"fase": 8, "etapa": "Due diligence", "pregunta": "¿Se justifica GO / NO-GO?", "output": "Decisión de inversión", "siguiente": "Ejecución o monitoreo"},
])

display(style_table(sim_funnel))

fig = go.Figure(go.Funnel(
    y=sim_funnel["etapa"],
    x=[35, 28, 15, 8, 5, 4, 3, 2],
    textinfo="value+percent initial",
    marker={"color": [CIBEST["gray"], CIBEST["gray"], CIBEST["yellow"], CIBEST["gold"], CIBEST["gold"], CIBEST["amber"], CIBEST["amber"], CIBEST["green"]]},
))
fig.update_layout(title="Embudo SIM: RADAR es la fase cuantitativa, no la decisión final", height=520)
fig.show()

insight_box(
    "Lectura ejecutiva del embudo",
    "RADAR reduce el universo hacia una shortlist analítica. La decisión de entrada exige fases posteriores: inteligencia cualitativa, capacidades internas, modo de entrada y due diligence."
)


,fase,etapa,pregunta,output,siguiente
0,1,Objetivo estratégico,¿Para qué se internacionaliza Cibest?,Tesis y criterios de decisión,"Define universo, pesos y tesis por línea"
1,2,Screening primario,¿Qué mercados son viables?,Mercados admisibles,Evita analizar mercados inviables
2,3,RADAR,¿Dónde tiene sentido profundizar?,"Ranking, perfiles, drivers, señales",Activa profundización cualitativa
3,4,Inteligencia cualitativa,¿Qué explica la oportunidad y el riesgo?,Diagnóstico sectorial y competitivo,Valida o ajusta la señal cuantitativa
4,5,Capacidades Cibest,¿Puede Cibest capturar la oportunidad?,Gap de capacidades,Ajusta priorización
5,6,Priorización ajustada,¿Qué mercados quedan en shortlist?,Candidatos perfilados,Define modo de entrada
6,7,Modo de entrada,¿Cómo entrar?,Build / buy / partner / digital,Prepara due diligence
7,8,Due diligence,¿Se justifica GO / NO-GO?,Decisión de inversión,Ejecución o monitoreo



## 3. Datos usados, universo evaluado y filtros

Esta sección valida el universo de datos disponible y documenta las exclusiones por calidad. La regla metodológica es deliberadamente conservadora: **primero se excluyen mercados con baja cobertura efectiva y luego se imputa lo residual**. Esto evita que un país con demasiados faltantes altere artificialmente los ideales de comparación del modelo multicriterio.


In [58]:

# -----------------------------------------------------------------------------
# Load real master data
# -----------------------------------------------------------------------------
MASTER_FILE = Path("../notebooks/data/reports/master_radar_cibest.xlsx")
if not MASTER_FILE.exists():
    raise FileNotFoundError("No se encontró master_radar_cibest.xlsx en el directorio actual.")

master = pd.read_excel(MASTER_FILE, sheet_name="master", engine="openpyxl")
required_cols = {"country_iso3", "year", "variable", "value", "source"}
missing_cols = required_cols - set(master.columns)
if missing_cols:
    raise ValueError(f"El master no tiene las columnas requeridas: {sorted(missing_cols)}")

master = master.copy()
master["country_iso3"] = master["country_iso3"].astype(str).str.strip()
master["variable"] = master["variable"].astype(str).str.strip()
master["year"] = pd.to_numeric(master["year"], errors="coerce")
master["value"] = pd.to_numeric(master["value"], errors="coerce")

country_count = master["country_iso3"].nunique()
variable_count = master["variable"].nunique()
year_min = int(master["year"].min())
year_max = int(master["year"].max())

# Latest observation by country-variable
latest = (
    master.dropna(subset=["value", "year"])
    .sort_values(["country_iso3", "variable", "year"])
    .groupby(["country_iso3", "variable"], as_index=False)
    .tail(1)
)
wide_latest = latest.pivot(index="country_iso3", columns="variable", values="value")

# Coverage diagnostics
coverage_country = wide_latest.notna().mean(axis=1).rename("coverage_pct").reset_index()
coverage_variable = wide_latest.notna().mean(axis=0).rename("coverage_pct").reset_index().rename(columns={"index": "variable"})
excluded_by_quality = coverage_country.query("coverage_pct < 0.70")["country_iso3"].tolist()

# Variables known to be excluded from TOPSIS because they feed IPC or Trend
excluded_from_topsis = [
    "colombian_diaspora_stock", "common_language_spanish", "cultural_distance_hofstede",
    "gdp_growth", "geographic_distance_km", "hofstede_idv", "hofstede_ivr",
    "hofstede_lto", "hofstede_mas", "hofstede_pdi", "hofstede_uai",
]

summary_data = pd.DataFrame({
    "métrica": [
        "Países en master", "Variables en master", "Años de referencia", "Países con cobertura <70%", "Variables excluidas de TOPSIS conocidas"
    ],
    "valor": [
        country_count, variable_count, f"{year_min}-{year_max}", ", ".join(excluded_by_quality) if excluded_by_quality else "Ninguno por regla <70%", len([v for v in excluded_from_topsis if v in master["variable"].unique()])
    ]
})

display(style_table(summary_data))

display(style_table(
    coverage_country.sort_values("coverage_pct").head(12),
    gradient_cols=["coverage_pct"],
    gradient_cmap="RdYlGn",
    format_dict={"coverage_pct": "{:.1%}"},
).set_caption("Cobertura efectiva por país — últimas observaciones disponibles"))

insight_box(
    "Lectura ejecutiva de calidad de datos",
    "Los países con baja cobertura no deben interpretarse como descartados estratégicamente. Su no priorización refleja que, con la información disponible y el horizonte actual, no son candidatos suficientemente comparables para Cibest."
)


,métrica,valor
0,Países en master,30
1,Variables en master,46
2,Años de referencia,1996-2026
3,Países con cobertura <70%,CUB
4,Variables excluidas de TOPSIS conocidas,10


,country_iso3,coverage_pct
10,CUB,52.2%
15,GUY,71.7%
17,HTI,73.9%
1,BHS,76.1%
5,BRB,78.3%
2,BLZ,82.6%
25,SUR,82.6%
20,NIC,82.6%
18,JAM,91.3%
16,HND,91.3%



## Validación crítica previa: consistencia entre Notebook 03 y Notebook 04

Antes de presentar recomendaciones, el notebook valida la consistencia conceptual entre resultados determinísticos y robustez:

- Notebook 03: identifica rankings, drivers, bandas, gaps y tesis por línea.
- Notebook 04: evalúa confiabilidad mediante sensibilidad determinística, comparación metodológica y Monte Carlo RADAR.
- Si ranking base y robustez divergen, la recomendación debe ser condicional, no concluyente.


In [59]:

critical_validation = pd.DataFrame([
    {"validación": "Sensibilidad determinística", "evidencia disponible": "Spearman medio reportado ≈ 0.995; Top-10 overlap medio ≈ 9.3", "lectura": "El ranking estructural no depende críticamente de perturbaciones puntuales de pesos."},
    {"validación": "TOPSIS vs VIKOR", "evidencia disponible": "Spearman reportado ≈ 0.932", "lectura": "La priorización es metodológicamente consistente entre dos técnicas multicriterio razonables."},
    {"validación": "Monte Carlo RADAR", "evidencia disponible": "Diseñado para 1.000 simulaciones, 24 países, 5 líneas, 120.000 observaciones país-línea-escenario", "lectura": "La prueba relevante de decisión es RADAR, no TOPSIS aislado."},
    {"validación": "Resultados vs recomendación", "evidencia disponible": "RADAR + drivers + robustez + gaps", "lectura": "Un país alto en ranking base pero sensible debe clasificarse como oportunidad condicional."},
])

display(style_table(critical_validation))

insight_box(
    "Conclusión de consistencia",
    "Los resultados determinísticos y la lógica de robustez son consistentes: el ranking base sirve como escenario central, mientras Monte Carlo determina la confiabilidad de avanzar a profundización."
)


,validación,evidencia disponible,lectura
0,Sensibilidad determinística,Spearman medio reportado ≈ 0.995; Top-10 overlap medio ≈ 9.3,El ranking estructural no depende críticamente de perturbaciones puntuales de pesos.
1,TOPSIS vs VIKOR,Spearman reportado ≈ 0.932,La priorización es metodológicamente consistente entre dos técnicas multicriterio razonables.
2,Monte Carlo RADAR,"Diseñado para 1.000 simulaciones, 24 países, 5 líneas, 120.000 observaciones país-línea-escenario","La prueba relevante de decisión es RADAR, no TOPSIS aislado."
3,Resultados vs recomendación,RADAR + drivers + robustez + gaps,Un país alto en ranking base pero sensible debe clasificarse como oportunidad condicional.



## 4. Ranking RADAR global: países más atractivos para profundización

El ranking RADAR global ordena mercados según su mezcla de atractivo estructural, proximidad con Colombia y momentum. Debe leerse como **shortlist de profundización**, no como recomendación de entrada.

**Estos son los países que deberían pasar primero a profundización cualitativa, de capacidades y modo de entrada.**


In [60]:

# -----------------------------------------------------------------------------
# Fallback actual results extracted from previous RADAR notebook outputs.
# If repo artifacts are available, replace these objects with live outputs.
# -----------------------------------------------------------------------------
radar_global = pd.DataFrame([
    {"country_iso3": "PAN", "radar_score": 0.691},
    {"country_iso3": "CRI", "radar_score": 0.653},
    {"country_iso3": "ESP", "radar_score": 0.627},
    {"country_iso3": "DOM", "radar_score": 0.620},
    {"country_iso3": "CHL", "radar_score": 0.569},
    {"country_iso3": "USA", "radar_score": 0.552},
    {"country_iso3": "URY", "radar_score": 0.539},
    {"country_iso3": "ECU", "radar_score": 0.538},
    {"country_iso3": "MEX", "radar_score": 0.533},
    {"country_iso3": "GTM", "radar_score": 0.530},
    {"country_iso3": "PER", "radar_score": 0.521},
    {"country_iso3": "SLV", "radar_score": 0.516},
    {"country_iso3": "HND", "radar_score": 0.513},
    {"country_iso3": "CAN", "radar_score": 0.506},
    {"country_iso3": "BHS", "radar_score": 0.493},
])
radar_global["rank"] = radar_global["radar_score"].rank(ascending=False, method="min").astype(int)

q80, q60, q40 = radar_global["radar_score"].quantile([0.80, 0.60, 0.40]).tolist()

def assign_tier(score: float) -> str:
    """Asigna banda de atractividad por cuantiles."""
    if score >= q80:
        return "Alta"
    if score >= q60:
        return "Media-alta"
    if score >= q40:
        return "Media"
    return "Baja"

radar_global["banda_atractividad"] = radar_global["radar_score"].apply(assign_tier)

display(style_table(
    radar_global,
    gradient_cols=["radar_score"],
    gradient_cmap="RdYlGn",
    format_dict={"radar_score": "{:.3f}", "rank": "{:.0f}"},
).set_caption("Top 15 RADAR global — mercados para profundización"))

fig = px.bar(
    radar_global.sort_values("radar_score"),
    x="radar_score",
    y="country_iso3",
    color="banda_atractividad",
    orientation="h",
    color_discrete_map=TIER_COLORS,
    title="Top 15 RADAR global: priorización inicial para profundización",
)
fig.update_layout(xaxis_title="Score RADAR", yaxis_title="País", height=560)
fig.show()

insight_box(
    "Lectura ejecutiva del ranking global",
    "Panamá, Costa Rica, España y República Dominicana lideran la lectura global porque combinan señales de atractivo, proximidad o momentum. Esta tabla no define entrada; define dónde enfocar primero la investigación cualitativa y la evaluación de capacidades."
)


,country_iso3,radar_score,rank,banda_atractividad
0,PAN,0.691,1,Alta
1,CRI,0.653,2,Alta
2,ESP,0.627,3,Alta
3,DOM,0.620,4,Media-alta
4,CHL,0.569,5,Media-alta
5,USA,0.552,6,Media-alta
6,URY,0.539,7,Media
7,ECU,0.538,8,Media
8,MEX,0.533,9,Media
9,GTM,0.530,10,Baja



## 5. Descomposición RADAR: estructura, proximidad y momentum

RADAR integra tres señales distintas:

- **TOPSIS:** atractivo estructural del país.
- **IPC:** proximidad y afinidad con Colombia.
- **Trend:** momentum macroeconómico reciente.

La utilidad ejecutiva de esta descomposición es evitar una lectura ingenua: un país estructuralmente fuerte puede bajar si es menos ejecutable para Cibest; un país cercano puede subir, pero requerir una tesis selectiva.


In [61]:

# -----------------------------------------------------------------------------
# Component-level fallback: known movements and IPC anchors from previous outputs.
# When live outputs are available, replace this with component_df from Notebook 03.
# -----------------------------------------------------------------------------
component_display = radar_global.copy()
component_display["topsis_score"] = np.nan
component_display["ipc"] = np.nan
component_display["trend"] = np.nan
component_display.loc[component_display["country_iso3"].eq("ECU"), "ipc"] = 0.954
component_display.loc[component_display["country_iso3"].eq("PAN"), "ipc"] = 0.942
component_display.loc[component_display["country_iso3"].eq("DOM"), "ipc"] = 0.864

known_delta = {
    "PAN": 7, "CRI": 4, "DOM": 7, "ECU": 12,
    "CAN": -12, "USA": -5, "BHS": -8, "JAM": -11,
}
component_display["delta_rank_radar_vs_topsis"] = component_display["country_iso3"].map(known_delta)

# Contribution columns are created only if components are available. Otherwise warn.
if component_display[["topsis_score", "ipc", "trend"]].notna().sum().sum() == 0:
    warn_missing("component_df completo", "se mostrarán movimientos conocidos y se deberá recalcular la descomposición TOPSIS/IPC/Trend desde Notebook 03 o el repositorio src/.")

raisers = component_display.dropna(subset=["delta_rank_radar_vs_topsis"]).sort_values("delta_rank_radar_vs_topsis", ascending=False).head(8)
fallers = component_display.dropna(subset=["delta_rank_radar_vs_topsis"]).sort_values("delta_rank_radar_vs_topsis").head(8)

movement_table = pd.concat([
    raisers.assign(tipo_movimiento="Sube en RADAR"),
    fallers.assign(tipo_movimiento="Baja en RADAR"),
], ignore_index=True)

display(style_table(
    movement_table[["country_iso3", "radar_score", "rank", "ipc", "delta_rank_radar_vs_topsis", "tipo_movimiento"]],
    gradient_cols=["radar_score", "ipc", "delta_rank_radar_vs_topsis"],
    gradient_cmap="RdYlGn",
    format_dict={"radar_score": "{:.3f}", "ipc": "{:.3f}", "delta_rank_radar_vs_topsis": "{:+.0f}"},
).set_caption("Movimientos estratégicos entre TOPSIS y RADAR"))

fig = px.scatter(
    component_display,
    x="ipc",
    y="radar_score",
    text="country_iso3",
    color="banda_atractividad",
    color_discrete_map=TIER_COLORS,
    title="Ejecutabilidad vs score RADAR: países cercanos no siempre son estructuralmente líderes",
)
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="IPC / proximidad con Colombia", yaxis_title="Score RADAR", height=520)
fig.show()

insight_box(
    "Lectura de movimientos",
    "Los países que suben al pasar de TOPSIS a RADAR suelen ganar por proximidad o momentum. Los que bajan pueden seguir siendo estructuralmente atractivos, pero menos ejecutables desde Colombia o menos alineados al horizonte actual."
)


,country_iso3,radar_score,rank,ipc,delta_rank_radar_vs_topsis,tipo_movimiento
0,ECU,0.538,8,0.954,+12,Sube en RADAR
1,PAN,0.691,1,0.942,+7,Sube en RADAR
2,DOM,0.620,4,0.864,+7,Sube en RADAR
3,CRI,0.653,2,nan,+4,Sube en RADAR
4,USA,0.552,6,nan,-5,Sube en RADAR
5,BHS,0.493,15,nan,-8,Sube en RADAR
6,CAN,0.506,14,nan,-12,Sube en RADAR
7,CAN,0.506,14,nan,-12,Baja en RADAR
8,BHS,0.493,15,nan,-8,Baja en RADAR
9,USA,0.552,6,nan,-5,Baja en RADAR


In [62]:

# -----------------------------------------------------------------------------
# Archetypes using available component proxies. Full classification should be recalculated when component_df is available.
# -----------------------------------------------------------------------------
archetype_map = {
    "CRI": "Prioridad natural Cibest",
    "PAN": "Cercano pero requiere tesis selectiva",
    "DOM": "Cercano pero requiere tesis selectiva",
    "PER": "Cercano pero requiere tesis selectiva",
    "SLV": "Cercano pero requiere tesis selectiva",
    "BOL": "Cercano pero requiere tesis selectiva",
    "ECU": "Cercano pero requiere tesis selectiva",
    "ESP": "Atractivo estructural lejano",
    "CHL": "Atractivo estructural lejano",
    "USA": "Atractivo estructural lejano",
    "URY": "Atractivo estructural lejano",
    "CAN": "Atractivo estructural lejano",
    "BHS": "Atractivo estructural lejano",
    "GTM": "Momentum táctico",
    "HND": "Momentum táctico",
}
component_display["arquetipo"] = component_display["country_iso3"].map(archetype_map).fillna("Monitoreo / oportunidad condicionada")

archetype_counts = component_display["arquetipo"].value_counts().reset_index()
archetype_counts.columns = ["arquetipo", "n_paises"]

display(style_table(archetype_counts, gradient_cols=["n_paises"], format_dict={"n_paises": "{:.0f}"}))

fig = px.scatter(
    component_display,
    x="rank",
    y="radar_score",
    color="arquetipo",
    text="country_iso3",
    title="Arquetipos estratégicos de mercados según lectura RADAR",
)
fig.update_traces(textposition="top center")
fig.update_layout(xaxis_title="Ranking RADAR", yaxis_title="Score RADAR", height=560, xaxis_autorange="reversed")
fig.show()

insight_box(
    "Lectura de arquetipos",
    "Los arquetipos evitan tratar todos los países del Top 15 igual. Costa Rica aparece como prioridad natural; Panamá y República Dominicana como mercados cercanos que requieren tesis selectiva; Estados Unidos, Canadá y España como mercados estructuralmente fuertes que exigen mayor validación de ejecutabilidad y capacidades."
)


,arquetipo,n_paises
0,Atractivo estructural lejano,6
1,Cercano pero requiere tesis selectiva,5
2,Momentum táctico,2
3,Prioridad natural Cibest,1
4,Monitoreo / oportunidad condicionada,1



## 6. Resultados por línea de negocio

Cada línea de negocio expresa una tesis distinta de internacionalización. La priorización debe responder **para qué negocio** un país es atractivo, no solo si el país aparece alto en el ranking global.


In [63]:

# -----------------------------------------------------------------------------
# Business line rankings: actual reported outputs from Notebook 03 summaries.
# These are deterministic scores from the RADAR/TOPSIS line-level model.
# -----------------------------------------------------------------------------
line_rankings_data: Dict[str, List[Tuple[str, float]]] = {
    "IB": [("USA", 0.728), ("ESP", 0.702), ("CAN", 0.690), ("CHL", 0.652), ("BHS", 0.605)],
    "PF": [("ESP", 0.674), ("CAN", 0.643), ("USA", 0.628), ("CHL", 0.599), ("URY", 0.557), ("BRA", 0.543), ("ARG", 0.533), ("CRI", 0.509), ("JAM", 0.492), ("SUR", 0.470), ("PAN", 0.468), ("DOM", 0.464)],
    "AD": [("USA", 0.801), ("CAN", 0.743), ("ESP", 0.652), ("CHL", 0.637), ("URY", 0.618), ("CRI", 0.552), ("BHS", 0.515), ("BLZ", 0.475), ("PAN", 0.470), ("ARG", 0.409), ("DOM", 0.405), ("JAM", 0.402)],
    "BD": [("USA", 0.798), ("ESP", 0.745), ("CAN", 0.724), ("CHL", 0.703), ("BRA", 0.638), ("URY", 0.622), ("ARG", 0.622), ("CRI", 0.582), ("MEX", 0.539), ("BHS", 0.531), ("DOM", 0.530), ("PER", 0.529)],
    "CIB": [("CAN", 0.731), ("USA", 0.705), ("ESP", 0.641), ("CHL", 0.640), ("BHS", 0.552), ("JAM", 0.525), ("TTO", 0.514), ("DOM", 0.508), ("URY", 0.504), ("BRA", 0.504), ("PAN", 0.476), ("MEX", 0.473)],
}

business_thesis: Dict[str, str] = {
    "IB": "Intermediación Bancaria: profundidad financiera, fondeo, cartera, solvencia bancaria, riesgo e institucionalidad.",
    "PF": "Pagos y Flujos: pagos digitales, remesas, transaccionalidad, apertura, canales y movimiento de dinero.",
    "AD": "Activos Digitales: regulación, seguridad jurídica, infraestructura digital segura, sofisticación financiera y control institucional.",
    "BD": "Bancos Digitales: escala retail, conectividad, bancarización, población urbana y adopción digital.",
    "CIB": "Corporate & Investment Banking: escala económica, mercado de capitales, IED, apertura, institucionalidad y profundidad corporativa.",
}

line_rankings = []
for line, rows in line_rankings_data.items():
    for rank, (country, score) in enumerate(rows, start=1):
        line_rankings.append({"business_line": line, "country_iso3": country, "score": score, "rank": rank})
line_rankings_df = pd.DataFrame(line_rankings)

for line in ["IB", "PF", "AD", "BD", "CIB"]:
    tmp = line_rankings_df.query("business_line == @line").sort_values("rank").head(10)
    display(Markdown(f"### {line} — {business_thesis[line]}"))
    display(style_table(
        tmp,
        gradient_cols=["score"],
        gradient_cmap="RdYlGn",
        format_dict={"score": "{:.3f}", "rank": "{:.0f}"},
    ))
    fig = px.bar(
        tmp.sort_values("score"),
        x="score",
        y="country_iso3",
        orientation="h",
        title=f"Top 10 {line}: países más atractivos para la tesis de negocio",
        color="score",
        color_continuous_scale="RdYlGn",
    )
    fig.update_layout(xaxis_title="Score por línea", yaxis_title="País", height=420)
    fig.show()
    insight_box(
        f"Lectura ejecutiva {line}",
        f"El ranking de {line} debe interpretarse bajo su tesis específica. Un país alto aquí no necesariamente es atractivo para todas las líneas; indica que sus fundamentos son más compatibles con los drivers de {line}."
    )


### IB — Intermediación Bancaria: profundidad financiera, fondeo, cartera, solvencia bancaria, riesgo e institucionalidad.

,business_line,country_iso3,score,rank
0,IB,USA,0.728,1
1,IB,ESP,0.702,2
2,IB,CAN,0.690,3
3,IB,CHL,0.652,4
4,IB,BHS,0.605,5


### PF — Pagos y Flujos: pagos digitales, remesas, transaccionalidad, apertura, canales y movimiento de dinero.

,business_line,country_iso3,score,rank
5,PF,ESP,0.674,1
6,PF,CAN,0.643,2
7,PF,USA,0.628,3
8,PF,CHL,0.599,4
9,PF,URY,0.557,5
10,PF,BRA,0.543,6
11,PF,ARG,0.533,7
12,PF,CRI,0.509,8
13,PF,JAM,0.492,9
14,PF,SUR,0.470,10


### AD — Activos Digitales: regulación, seguridad jurídica, infraestructura digital segura, sofisticación financiera y control institucional.

,business_line,country_iso3,score,rank
17,AD,USA,0.801,1
18,AD,CAN,0.743,2
19,AD,ESP,0.652,3
20,AD,CHL,0.637,4
21,AD,URY,0.618,5
22,AD,CRI,0.552,6
23,AD,BHS,0.515,7
24,AD,BLZ,0.475,8
25,AD,PAN,0.470,9
26,AD,ARG,0.409,10


### BD — Bancos Digitales: escala retail, conectividad, bancarización, población urbana y adopción digital.

,business_line,country_iso3,score,rank
29,BD,USA,0.798,1
30,BD,ESP,0.745,2
31,BD,CAN,0.724,3
32,BD,CHL,0.703,4
33,BD,BRA,0.638,5
34,BD,URY,0.622,6
35,BD,ARG,0.622,7
36,BD,CRI,0.582,8
37,BD,MEX,0.539,9
38,BD,BHS,0.531,10


### CIB — Corporate & Investment Banking: escala económica, mercado de capitales, IED, apertura, institucionalidad y profundidad corporativa.

,business_line,country_iso3,score,rank
41,CIB,CAN,0.731,1
42,CIB,USA,0.705,2
43,CIB,ESP,0.641,3
44,CIB,CHL,0.640,4
45,CIB,BHS,0.552,5
46,CIB,JAM,0.525,6
47,CIB,TTO,0.514,7
48,CIB,DOM,0.508,8
49,CIB,URY,0.504,9
50,CIB,BRA,0.504,10



## 7. Países transversalmente atractivos vs apuestas especializadas

Un país transversalmente atractivo aparece alto en varias líneas; una apuesta especializada aparece fuerte solo en una tesis específica. Esta distinción es crítica para decidir si el Comité debe discutir una agenda multi-línea o una profundización focalizada.


In [64]:

# -----------------------------------------------------------------------------
# Cross-line heatmap and dispersion
# -----------------------------------------------------------------------------
score_matrix = line_rankings_df.pivot_table(index="country_iso3", columns="business_line", values="score", aggfunc="max")
rank_matrix = line_rankings_df.pivot_table(index="country_iso3", columns="business_line", values="rank", aggfunc="min")

# Fill absent scores with NaN to avoid inventing scores.
ordered_countries = score_matrix.mean(axis=1, skipna=True).sort_values(ascending=False).index.tolist()
score_matrix = score_matrix.loc[ordered_countries]

fig = px.imshow(
    score_matrix,
    color_continuous_scale="RdYlGn",
    aspect="auto",
    title="Heatmap país x línea: atractivo relativo por tesis de negocio",
)
fig.update_layout(xaxis_title="Línea de negocio", yaxis_title="País", height=650)
fig.show()

# Dispersion only where at least 3 line scores exist
rank_dispersion = rank_matrix.copy()
rank_dispersion["n_lines_observed"] = rank_dispersion.notna().sum(axis=1)
rank_dispersion["mean_rank"] = rank_matrix.mean(axis=1, skipna=True)
rank_dispersion["std_rank"] = rank_matrix.std(axis=1, skipna=True)
rank_dispersion["rank_range"] = rank_matrix.max(axis=1, skipna=True) - rank_matrix.min(axis=1, skipna=True)
rank_dispersion = rank_dispersion.reset_index().sort_values(["rank_range", "mean_rank"], ascending=[True, True])

stable_multi = rank_dispersion.query("n_lines_observed >= 3").head(10)
specialized = rank_dispersion.query("n_lines_observed >= 3").sort_values("rank_range", ascending=False).head(10)

display(style_table(
    stable_multi,
    gradient_cols=["mean_rank", "std_rank", "rank_range", "n_lines_observed"],
    gradient_cmap="RdYlGn",
    format_dict={"mean_rank": "{:.1f}", "std_rank": "{:.1f}", "rank_range": "{:.0f}", "n_lines_observed": "{:.0f}"},
).set_caption("Países con atractivo transversal más estable"))

display(style_table(
    specialized,
    gradient_cols=["mean_rank", "std_rank", "rank_range", "n_lines_observed"],
    gradient_cmap="YlOrRd",
    format_dict={"mean_rank": "{:.1f}", "std_rank": "{:.1f}", "rank_range": "{:.0f}", "n_lines_observed": "{:.0f}"},
).set_caption("Países más dependientes de tesis específica"))

insight_box(
    "Lectura de transversalidad",
    "Estados Unidos, Canadá, España y Chile aparecen de forma recurrente en varias líneas, lo que sugiere atractivo transversal. Otros países pueden ser valiosos solo para una tesis específica y deben profundizarse de forma focalizada."
)


business_line,country_iso3,AD,BD,CIB,IB,PF,n_lines_observed,mean_rank,std_rank,rank_range
5,CHL,4.000000,4.000000,4.000000,4.000000,4.000000,5,4.0,0.0,0
16,USA,1.000000,1.000000,2.000000,1.000000,3.000000,5,1.6,0.9,2
4,CAN,2.000000,3.000000,1.000000,3.000000,2.000000,5,2.2,0.8,2
8,ESP,3.000000,2.000000,3.000000,2.000000,1.000000,5,2.2,0.8,2
6,CRI,6.000000,8.000000,nan,nan,8.000000,3,7.3,1.2,2
11,PAN,9.000000,nan,11.000000,nan,11.000000,3,10.3,1.2,2
0,ARG,10.000000,7.000000,nan,nan,7.000000,3,8.0,1.7,3
15,URY,5.000000,6.000000,9.000000,nan,5.000000,4,6.2,1.9,4
7,DOM,11.000000,11.000000,8.000000,nan,12.000000,4,10.5,1.7,4
1,BHS,7.000000,10.000000,5.000000,5.000000,nan,4,6.8,2.4,5


business_line,country_iso3,AD,BD,CIB,IB,PF,n_lines_observed,mean_rank,std_rank,rank_range
9,JAM,12.000000,nan,6.000000,nan,9.000000,3,9.0,3.0,6
1,BHS,7.000000,10.000000,5.000000,5.000000,nan,4,6.8,2.4,5
3,BRA,nan,5.000000,10.000000,nan,6.000000,3,7.0,2.6,5
15,URY,5.000000,6.000000,9.000000,nan,5.000000,4,6.2,1.9,4
7,DOM,11.000000,11.000000,8.000000,nan,12.000000,4,10.5,1.7,4
0,ARG,10.000000,7.000000,nan,nan,7.000000,3,8.0,1.7,3
16,USA,1.000000,1.000000,2.000000,1.000000,3.000000,5,1.6,0.9,2
4,CAN,2.000000,3.000000,1.000000,3.000000,2.000000,5,2.2,0.8,2
8,ESP,3.000000,2.000000,3.000000,2.000000,1.000000,5,2.2,0.8,2
6,CRI,6.000000,8.000000,nan,nan,8.000000,3,7.3,1.2,2


In [65]:

# -----------------------------------------------------------------------------
# Top-N overlap by line
# -----------------------------------------------------------------------------
def topn_overlap(df: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    """Calcula solapamiento Top-N entre líneas de negocio."""
    rows: List[Dict[str, Any]] = []
    lines = sorted(df["business_line"].unique())
    for i, line_a in enumerate(lines):
        for line_b in lines[i + 1:]:
            a = df.query("business_line == @line_a").sort_values("rank").head(n)["country_iso3"].tolist()
            b = df.query("business_line == @line_b").sort_values("rank").head(n)["country_iso3"].tolist()
            overlap = len(set(a) & set(b))
            rows.append({
                "line_a": line_a,
                "line_b": line_b,
                "n": n,
                "overlap": overlap,
                "overlap_pct": overlap / n,
                "top_a": ", ".join(a),
                "top_b": ", ".join(b),
            })
    return pd.DataFrame(rows).sort_values("overlap_pct", ascending=False)

top5_overlap = topn_overlap(line_rankings_df, n=5)
top10_overlap = topn_overlap(line_rankings_df, n=10)

display(style_table(top5_overlap, gradient_cols=["overlap_pct"], gradient_cmap="YlOrRd", format_dict={"overlap_pct": "{:.0%}"}).set_caption("Solapamiento Top 5 entre líneas"))
display(style_table(top10_overlap, gradient_cols=["overlap_pct"], gradient_cmap="YlOrRd", format_dict={"overlap_pct": "{:.0%}"}).set_caption("Solapamiento Top 10 entre líneas"))

insight_box(
    "Atractivo común no es redundancia automática",
    "Un alto solapamiento Top-N puede indicar que ciertos países concentran condiciones transversales —institucionalidad, digitalización, profundidad financiera y estabilidad— que son valiosas para varias líneas. La redundancia solo sería problemática si las tesis y pesos fueran idénticos."
)


,line_a,line_b,n,overlap,overlap_pct,top_a,top_b
3,AD,PF,5,5,100%,"USA, CAN, ESP, CHL, URY","ESP, CAN, USA, CHL, URY"
7,CIB,IB,5,5,100%,"CAN, USA, ESP, CHL, BHS","USA, ESP, CAN, CHL, BHS"
0,AD,BD,5,4,80%,"USA, CAN, ESP, CHL, URY","USA, ESP, CAN, CHL, BRA"
1,AD,CIB,5,4,80%,"USA, CAN, ESP, CHL, URY","CAN, USA, ESP, CHL, BHS"
2,AD,IB,5,4,80%,"USA, CAN, ESP, CHL, URY","USA, ESP, CAN, CHL, BHS"
4,BD,CIB,5,4,80%,"USA, ESP, CAN, CHL, BRA","CAN, USA, ESP, CHL, BHS"
5,BD,IB,5,4,80%,"USA, ESP, CAN, CHL, BRA","USA, ESP, CAN, CHL, BHS"
6,BD,PF,5,4,80%,"USA, ESP, CAN, CHL, BRA","ESP, CAN, USA, CHL, URY"
8,CIB,PF,5,4,80%,"CAN, USA, ESP, CHL, BHS","ESP, CAN, USA, CHL, URY"
9,IB,PF,5,4,80%,"USA, ESP, CAN, CHL, BHS","ESP, CAN, USA, CHL, URY"


,line_a,line_b,n,overlap,overlap_pct,top_a,top_b
0,AD,BD,10,8,80%,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS"
6,BD,PF,10,8,80%,"USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS","ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR"
3,AD,PF,10,7,70%,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR"
4,BD,CIB,10,7,70%,"USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS","CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA"
8,CIB,PF,10,7,70%,"CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA","ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR"
1,AD,CIB,10,6,60%,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA"
2,AD,IB,10,5,50%,"USA, CAN, ESP, CHL, URY, CRI, BHS, BLZ, PAN, ARG","USA, ESP, CAN, CHL, BHS"
5,BD,IB,10,5,50%,"USA, ESP, CAN, CHL, BRA, URY, ARG, CRI, MEX, BHS","USA, ESP, CAN, CHL, BHS"
7,CIB,IB,10,5,50%,"CAN, USA, ESP, CHL, BHS, JAM, TTO, DOM, URY, BRA","USA, ESP, CAN, CHL, BHS"
9,IB,PF,10,4,40%,"USA, ESP, CAN, CHL, BHS","ESP, CAN, USA, CHL, URY, BRA, ARG, CRI, JAM, SUR"



## 8. Variables que explican el atractivo

Esta sección conecta los países priorizados con variables observables. El objetivo es responder: **¿qué hace atractivo a un país y qué restricciones deben validarse antes de avanzar?**

Se usan variables reales del `master_radar_cibest.xlsx`, tomando la última observación disponible por país-variable. Cuando los objetos de contribución/marginal de Notebook 03 estén disponibles, el notebook puede reemplazar este diagnóstico descriptivo por drivers robustos y restricciones críticas calculadas formalmente.


In [66]:

# -----------------------------------------------------------------------------
# Key variable heatmap from real master data
# -----------------------------------------------------------------------------
key_variables = [
    "digital_payment_adults_pct", "account_ownership", "internet_users_pct", "mobile_subscriptions",
    "stock_market_cap_gdp", "regulatory_quality", "rule_of_law", "control_of_corruption",
    "secure_internet_servers_per_million", "domestic_credit_private_gdp", "financial_system_deposits_gdp",
    "bank_npl_ratio", "country_risk_premium", "public_debt_gdp", "personal_remittances_gdp",
    "trade_openness", "gdp_nominal", "gdp_per_capita_ppp",
]

available_key_vars = [v for v in key_variables if v in wide_latest.columns]
top_countries_union = sorted(set(radar_global.head(10)["country_iso3"]) | set(line_rankings_df.query("rank <= 5")["country_iso3"]))

raw_key = wide_latest.reindex(index=top_countries_union, columns=available_key_vars)

# Normalize for visualization only. Directionality is not applied here, except for known cost/risk variables.
higher_is_worse = {"bank_npl_ratio", "country_risk_premium", "public_debt_gdp"}
normalized_key = raw_key.copy()
for col in normalized_key.columns:
    series = pd.to_numeric(normalized_key[col], errors="coerce")
    mn, mx = series.min(), series.max()
    if pd.notna(mn) and pd.notna(mx) and mx > mn:
        normalized_key[col] = (series - mn) / (mx - mn)
        if col in higher_is_worse:
            normalized_key[col] = 1 - normalized_key[col]
    else:
        normalized_key[col] = np.nan

fig = px.imshow(
    normalized_key,
    color_continuous_scale="RdYlGn",
    aspect="auto",
    title="Variables reales que explican atractivo en países priorizados — normalizadas para comparación visual",
)
fig.update_layout(height=720, xaxis_title="Variable", yaxis_title="País")
fig.show()

insight_box(
    "Lectura de variables clave",
    "La visualización no reemplaza TOPSIS, pero ayuda a explicar qué atributos concretos sostienen el atractivo: digitalización, bancarización, institucionalidad, profundidad financiera y riesgo. Las variables de riesgo se invierten para que verde indique mejor posición relativa."
)


In [67]:

# -----------------------------------------------------------------------------
# Weight spread fallback from previous Notebook 03 diagnostics.
# -----------------------------------------------------------------------------
weight_spread_fallback = pd.DataFrame([
    {"variable": "digital_payment_adults_pct", "dominant_line": "PF", "spread": 0.1860, "lectura": "Diferencia la tesis de Pagos y Flujos por adopción transaccional."},
    {"variable": "stock_market_cap_gdp", "dominant_line": "CIB", "spread": 0.1415, "lectura": "Diferencia CIB por profundidad de mercado de capitales."},
    {"variable": "regulatory_quality", "dominant_line": "AD", "spread": 0.1356, "lectura": "Diferencia Activos Digitales por calidad regulatoria."},
    {"variable": "internet_users_pct", "dominant_line": "BD", "spread": 0.0503, "lectura": "Refuerza tesis de escala digital y adquisición."},
    {"variable": "account_ownership", "dominant_line": "BD", "spread": 0.0300, "lectura": "Base potencial de clientes financieros digitales."},
    {"variable": "secure_internet_servers_per_million", "dominant_line": "AD", "spread": 0.0245, "lectura": "Infraestructura digital segura y capacidad operativa avanzada."},
])

display(style_table(
    weight_spread_fallback,
    gradient_cols=["spread"],
    gradient_cmap="YlOrRd",
    format_dict={"spread": "{:.4f}"},
).set_caption("Variables diferenciadoras de tesis por línea"))

fig = px.bar(
    weight_spread_fallback.sort_values("spread"),
    x="spread",
    y="variable",
    color="dominant_line",
    orientation="h",
    title="Variables que más separan las tesis de negocio",
)
fig.update_layout(xaxis_title="Spread de peso entre líneas", yaxis_title="Variable", height=480)
fig.show()

insight_box(
    "Lectura de diferenciadores",
    "Las variables diferenciadoras muestran que el modelo no es un ranking único disfrazado: Pagos se separa por transaccionalidad digital, CIB por mercado de capitales y Activos Digitales por regulación e infraestructura segura."
)


,variable,dominant_line,spread,lectura
0,digital_payment_adults_pct,PF,0.1860,Diferencia la tesis de Pagos y Flujos por adopción transaccional.
1,stock_market_cap_gdp,CIB,0.1415,Diferencia CIB por profundidad de mercado de capitales.
2,regulatory_quality,AD,0.1356,Diferencia Activos Digitales por calidad regulatoria.
3,internet_users_pct,BD,0.0503,Refuerza tesis de escala digital y adquisición.
4,account_ownership,BD,0.0300,Base potencial de clientes financieros digitales.
5,secure_internet_servers_per_million,AD,0.0245,Infraestructura digital segura y capacidad operativa avanzada.


In [68]:

# -----------------------------------------------------------------------------
# Driver/restriction qualitative table using actual known diagnostics from Notebook 03 summaries.
# -----------------------------------------------------------------------------
drivers_known = pd.DataFrame([
    {"country_iso3": "USA", "business_line": "IB", "driver": "financial_system_deposits_gdp", "tipo": "Driver robusto", "lectura": "Profundidad de fondeo y sistema financiero amplio."},
    {"country_iso3": "USA", "business_line": "IB", "driver": "regulatory_quality", "tipo": "Driver robusto", "lectura": "Calidad regulatoria sostiene operación bancaria."},
    {"country_iso3": "USA", "business_line": "AD", "driver": "stock_market_cap_gdp", "tipo": "Driver robusto", "lectura": "Sofisticación financiera relevante para activos digitales."},
    {"country_iso3": "ESP", "business_line": "PF", "driver": "digital_payment_adults_pct", "tipo": "Driver robusto", "lectura": "Alta adopción de pagos digitales sostiene la tesis de flujos."},
    {"country_iso3": "ESP", "business_line": "PF", "driver": "personal_remittances_gdp", "tipo": "Restricción crítica", "lectura": "La baja relevancia de remesas puede limitar la tesis de corredor."},
    {"country_iso3": "CAN", "business_line": "CIB", "driver": "rule_of_law", "tipo": "Driver robusto", "lectura": "Seguridad jurídica y entorno institucional favorable."},
    {"country_iso3": "CAN", "business_line": "CIB", "driver": "stock_market_cap_gdp", "tipo": "Driver robusto", "lectura": "Profundidad de mercado de capitales para banca corporativa e inversión."},
    {"country_iso3": "USA", "business_line": "BD", "driver": "mobile_subscriptions", "tipo": "Restricción crítica", "lectura": "Puede penalizar comparativamente si otros mercados tienen mayor capilaridad móvil relativa."},
    {"country_iso3": "USA", "business_line": "AD", "driver": "public_debt_gdp", "tipo": "Restricción crítica", "lectura": "Riesgo macro-fiscal que debe ser revisado antes de cualquier tesis de entrada."},
])

display(style_table(drivers_known).set_caption("Drivers y restricciones conocidas de países prioritarios"))

fig = px.histogram(
    drivers_known,
    x="business_line",
    color="tipo",
    barmode="group",
    title="Drivers robustos y restricciones críticas identificadas en países prioritarios",
    color_discrete_map={"Driver robusto": CIBEST["green"], "Restricción crítica": CIBEST["red"]},
)
fig.update_layout(xaxis_title="Línea de negocio", yaxis_title="Número de señales", height=420)
fig.show()

insight_box(
    "Lectura de drivers",
    "Los drivers permiten pasar de ‘país atractivo’ a ‘tesis defendible’. Las restricciones críticas no descartan automáticamente un país, pero definen las preguntas que deben resolverse en inteligencia cualitativa y due diligence."
)


,country_iso3,business_line,driver,tipo,lectura
0,USA,IB,financial_system_deposits_gdp,Driver robusto,Profundidad de fondeo y sistema financiero amplio.
1,USA,IB,regulatory_quality,Driver robusto,Calidad regulatoria sostiene operación bancaria.
2,USA,AD,stock_market_cap_gdp,Driver robusto,Sofisticación financiera relevante para activos digitales.
3,ESP,PF,digital_payment_adults_pct,Driver robusto,Alta adopción de pagos digitales sostiene la tesis de flujos.
4,ESP,PF,personal_remittances_gdp,Restricción crítica,La baja relevancia de remesas puede limitar la tesis de corredor.
5,CAN,CIB,rule_of_law,Driver robusto,Seguridad jurídica y entorno institucional favorable.
6,CAN,CIB,stock_market_cap_gdp,Driver robusto,Profundidad de mercado de capitales para banca corporativa e inversión.
7,USA,BD,mobile_subscriptions,Restricción crítica,Puede penalizar comparativamente si otros mercados tienen mayor capilaridad móvil relativa.
8,USA,AD,public_debt_gdp,Restricción crítica,Riesgo macro-fiscal que debe ser revisado antes de cualquier tesis de entrada.



## 9. Robustez del modelo y probabilidades Top-N

**Monte Carlo no prueba éxito comercial. Prueba estabilidad de la recomendación ante incertidumbre razonable de pesos.**

La robustez debe leerse en cuatro dimensiones:

1. Probabilidad de permanecer en Top 3 / Top 5 / Top 10.
2. Ranking medio bajo escenarios simulados.
3. Volatilidad de ranking.
4. Probabilidad de permanecer en banda Alta o Media-alta.

Si los outputs exactos de Monte Carlo RADAR están disponibles, esta sección los carga. Si no están disponibles, deja el código preparado para recalcularlos desde el repositorio `src/`.


In [69]:

# -----------------------------------------------------------------------------
# Monte Carlo expected parameters and fallback warning.
# -----------------------------------------------------------------------------
mc_parameters = pd.DataFrame([
    {"parámetro": "Simulaciones", "valor": "1.000"},
    {"parámetro": "Países", "valor": "24"},
    {"parámetro": "Líneas", "valor": "5"},
    {"parámetro": "Observaciones país-línea-escenario", "valor": "120.000"},
    {"parámetro": "Perturbación pesos dimensión", "valor": "Sí"},
    {"parámetro": "Perturbación pesos variable", "valor": "Sí"},
    {"parámetro": "Perturbación alpha/beta/gamma", "valor": "Sí"},
])

display(style_table(mc_parameters).set_caption("Parámetros esperados de Monte Carlo RADAR"))

# Placeholder loader: if files with Monte Carlo artifacts exist, load them. Otherwise create an explicit warning table.
possible_mc_files = list(Path('.').glob('*monte*carlo*.csv')) + list(Path('.').glob('*mc_radar*.parquet')) + list(Path('.').glob('*robustness*.csv'))
if possible_mc_files:
    success_box("Artefactos Monte Carlo detectados", f"Se detectaron posibles archivos: {', '.join(p.name for p in possible_mc_files[:5])}. Ajuste esta celda para cargar el artefacto correcto.")
else:
    risk_box(
        "Probabilidades Top-N exactas no cargadas",
        "No se encontraron artefactos persistidos de Monte Carlo en el directorio actual. El Notebook 04 reporta la estructura de simulación; para obtener probabilidades exactas, ejecute la sección de recalculo contra el repositorio src/ o exporte mc_radar['topn_probabilities'], mc_radar['rank_robustness'] y mc_radar['tier_probabilities']."
    )

# Recalculation scaffold. It is disabled by default to avoid accidental heavy runs outside the repo.
RECALCULATE_MONTE_CARLO = False

if RECALCULATE_MONTE_CARLO:
    # This block requires the RADAR repository structure and src/ modules.
    sys.path.insert(0, str(Path.cwd().parent))
    from src.utils import load_all_configs, get_variable_catalog
    from src.scoring.hybrid_scorer import prepare_decision_matrix, run_full_scoring
    from src.scoring.monte_carlo import coerce_component_series, run_monte_carlo_radar_robustness

    configs = load_all_configs()
    variable_catalog = get_variable_catalog(configs["variables"])
    wide_raw_repo, decision_matrix_repo, excluded_countries_repo = prepare_decision_matrix(master, configs)
    results_repo = run_full_scoring(master, configs, persist=True)
    ipc_scores_repo = coerce_component_series(results_repo["ipc"], value_col=None, component_name="ipc")
    trend_scores_repo = coerce_component_series(results_repo["trend"], value_col="trend", component_name="trend")
    mc_cfg = configs["settings"].get("monte_carlo", {})
    mc_radar = run_monte_carlo_radar_robustness(
        decision_matrix=decision_matrix_repo,
        configs=configs,
        variable_catalog=variable_catalog,
        ipc_scores=ipc_scores_repo,
        trend_scores=trend_scores_repo,
        n_simulations=int(mc_cfg.get("n_simulations", 1000)),
        dimension_concentration=float(mc_cfg.get("topsis", {}).get("dimension_concentration", 150)),
        variable_concentration=float(mc_cfg.get("topsis", {}).get("variable_concentration", 100)),
        composite_concentration=float(mc_cfg.get("radar", {}).get("composite_concentration", 150)),
        perturb_composite_weights=bool(mc_cfg.get("radar", {}).get("perturb_composite_weights", True)),
        random_seed=int(mc_cfg.get("random_seed", 42)),
    )
    radar_topn = mc_radar["topn_probabilities"]
    radar_rank_robustness = mc_radar["rank_robustness"]
    radar_tiers = mc_radar["tier_probabilities"]
else:
    radar_topn = pd.DataFrame(columns=["business_line", "country_iso3", "prob_top_3", "prob_top_5", "prob_top_10"])
    radar_rank_robustness = pd.DataFrame(columns=["business_line", "country_iso3", "mean_rank", "std_rank", "p10_rank", "p90_rank"])
    radar_tiers = pd.DataFrame(columns=["business_line", "country_iso3", "Alta", "Media-alta", "Media", "Baja"])


,parámetro,valor
0,Simulaciones,1.000
1,Países,24
2,Líneas,5
3,Observaciones país-línea-escenario,120.000
4,Perturbación pesos dimensión,Sí
5,Perturbación pesos variable,Sí
6,Perturbación alpha/beta/gamma,Sí


In [70]:

# -----------------------------------------------------------------------------
# If Monte Carlo outputs are available, classify robustness. Otherwise show guidance.
# -----------------------------------------------------------------------------
def classify_robustness(row: pd.Series) -> str:
    """Clasifica la robustez ejecutiva país-línea."""
    prob_top_5 = row.get("prob_top_5", 0.0) if pd.notna(row.get("prob_top_5", np.nan)) else 0.0
    prob_top_10 = row.get("prob_top_10", 0.0) if pd.notna(row.get("prob_top_10", np.nan)) else 0.0
    prob_alta = row.get("Alta", 0.0) if pd.notna(row.get("Alta", np.nan)) else 0.0
    prob_alta_media = prob_alta + (row.get("Media-alta", 0.0) if pd.notna(row.get("Media-alta", np.nan)) else 0.0)
    std_rank = row.get("std_rank", np.nan)
    if prob_top_5 >= 0.80 and prob_alta >= 0.70:
        return "Prioridad robusta"
    if prob_top_10 >= 0.70 and prob_alta_media >= 0.70:
        return "Prioridad condicional"
    if pd.notna(std_rank) and std_rank >= 4:
        return "Alta sensibilidad"
    return "No robusta / pendiente evidencia"

if not radar_topn.empty:
    robustness_exec = (
        radar_rank_robustness
        .merge(radar_topn, on=["business_line", "country_iso3"], how="outer")
        .merge(radar_tiers, on=["business_line", "country_iso3"], how="outer")
    )
    robustness_exec["robustness_class"] = robustness_exec.apply(classify_robustness, axis=1)
    display(style_table(
        robustness_exec.sort_values(["business_line", "prob_top_5"], ascending=[True, False]).head(50),
        gradient_cols=["prob_top_5", "prob_top_10", "std_rank", "Alta", "Media-alta"],
        gradient_cmap="RdYlGn",
        format_dict={"prob_top_5": "{:.1%}", "prob_top_10": "{:.1%}", "std_rank": "{:.2f}", "Alta": "{:.1%}", "Media-alta": "{:.1%}"},
    ).set_caption("Robustez Monte Carlo RADAR — país/línea"))
else:
    robustness_exec = pd.DataFrame()
    mc_guidance = pd.DataFrame([
        {"métrica": "prob_top_5", "uso ejecutivo": "Mide probabilidad de permanecer en shortlist prioritaria."},
        {"métrica": "prob_top_10", "uso ejecutivo": "Mide opción defendible para shortlist ampliada."},
        {"métrica": "std_rank", "uso ejecutivo": "Mide volatilidad; alto valor implica sensibilidad."},
        {"métrica": "Alta / Media-alta", "uso ejecutivo": "Mide estabilidad de banda de atractividad."},
    ])
    display(style_table(mc_guidance).set_caption("Cómo interpretar probabilidades Monte Carlo cuando se carguen los artefactos"))

insight_box(
    "Lectura de robustez",
    "La recomendación ejecutiva debe privilegiar países que combinen buen ranking base, probabilidad Top-N alta y baja volatilidad. Si un país lidera el ranking pero es inestable en Monte Carlo, debe clasificarse como oportunidad condicional."
)


,métrica,uso ejecutivo
0,prob_top_5,Mide probabilidad de permanecer en shortlist prioritaria.
1,prob_top_10,Mide opción defendible para shortlist ampliada.
2,std_rank,Mide volatilidad; alto valor implica sensibilidad.
3,Alta / Media-alta,Mide estabilidad de banda de atractividad.



## 10. Bandas, gaps y materialidad de diferencias

No todas las diferencias de ranking son estratégicamente significativas. Si dos países tienen scores cercanos, deben comunicarse como una banda competitiva y no como una jerarquía fuerte.


In [71]:

def classify_gap(gap: float) -> str:
    """Clasifica la materialidad del gap de score."""
    if pd.isna(gap):
        return "Sin siguiente país"
    if gap < 0.005:
        return "Empate práctico"
    if gap < 0.015:
        return "Diferencia débil"
    if gap < 0.030:
        return "Diferencia moderada"
    return "Diferencia material"

radar_gap = radar_global.sort_values("radar_score", ascending=False).copy()
radar_gap["score_gap_next"] = radar_gap["radar_score"] - radar_gap["radar_score"].shift(-1)
radar_gap["gap_interpretation"] = radar_gap["score_gap_next"].apply(classify_gap)

display(style_table(
    radar_gap,
    gradient_cols=["radar_score", "score_gap_next"],
    gradient_cmap="RdYlGn",
    format_dict={"radar_score": "{:.3f}", "score_gap_next": "{:.3f}"},
).set_caption("Bandas y gaps del ranking RADAR global"))

fig = px.bar(
    radar_gap.sort_values("score_gap_next", ascending=True),
    x="score_gap_next",
    y="country_iso3",
    color="gap_interpretation",
    orientation="h",
    title="Materialidad del gap frente al siguiente país",
)
fig.update_layout(xaxis_title="Gap de score", yaxis_title="País", height=560)
fig.show()

insight_box(
    "Lectura de gaps",
    "El Comité debe evitar convertir diferencias pequeñas en narrativas de superioridad. Cuando el gap es débil o hay empate práctico, la discusión debe migrar a drivers, restricciones, robustez y capacidades."
)


,country_iso3,radar_score,rank,banda_atractividad,score_gap_next,gap_interpretation
0,PAN,0.691,1,Alta,0.038,Diferencia material
1,CRI,0.653,2,Alta,0.026,Diferencia moderada
2,ESP,0.627,3,Alta,0.007,Diferencia débil
3,DOM,0.620,4,Media-alta,0.051,Diferencia material
4,CHL,0.569,5,Media-alta,0.017,Diferencia moderada
5,USA,0.552,6,Media-alta,0.013,Diferencia débil
6,URY,0.539,7,Media,0.001,Empate práctico
7,ECU,0.538,8,Media,0.005,Diferencia débil
8,MEX,0.533,9,Media,0.003,Empate práctico
9,GTM,0.530,10,Baja,0.009,Diferencia débil



## 11. Recomendación ejecutiva: qué países deberían avanzar

La recomendación no debe ser “entrar a X país”. Debe ser:

> “X país debe avanzar a la siguiente fase del proceso SIM para evaluar la tesis Y, porque presenta alta atractividad relativa, drivers consistentes, robustez suficiente y restricciones identificables que pueden validarse en la fase cualitativa.”


In [72]:

# -----------------------------------------------------------------------------
# Executive recommendation matrix
# -----------------------------------------------------------------------------
# Top line presence
line_presence = (
    line_rankings_df.assign(
        top5=lambda d: d["rank"] <= 5,
        top10=lambda d: d["rank"] <= 10,
    )
    .groupby("country_iso3")
    .agg(
        lines_top5=("business_line", lambda s: ", ".join(sorted(line_rankings_df.loc[s.index].query("rank <= 5")["business_line"].unique()))),
        lines_top10=("business_line", lambda s: ", ".join(sorted(line_rankings_df.loc[s.index].query("rank <= 10")["business_line"].unique()))),
        n_lines_top5=("top5", "sum"),
        n_lines_top10=("top10", "sum"),
    )
    .reset_index()
)

recommendation = radar_gap.merge(line_presence, on="country_iso3", how="left")
recommendation["n_lines_top5"] = recommendation["n_lines_top5"].fillna(0).astype(int)
recommendation["n_lines_top10"] = recommendation["n_lines_top10"].fillna(0).astype(int)
recommendation["lines_top5"] = recommendation["lines_top5"].fillna("")
recommendation["lines_top10"] = recommendation["lines_top10"].fillna("")


def classify_action(row: pd.Series) -> str:
    """Clasifica la acción recomendada para el Comité."""
    if row["banda_atractividad"] == "Alta" and row["n_lines_top5"] >= 2:
        return "Avanzar a profundización"
    if row["banda_atractividad"] in {"Alta", "Media-alta"} and row["n_lines_top10"] >= 2:
        return "Mantener en shortlist"
    if row["banda_atractividad"] in {"Media", "Media-alta"}:
        return "Monitorear"
    return "No priorizar en horizonte actual"

recommendation["clasificación"] = recommendation.apply(classify_action, axis=1)
recommendation["próxima_fase_SIM"] = recommendation["clasificación"].map({
    "Avanzar a profundización": "Avanzar a Fase 4 — Inteligencia cualitativa",
    "Mantener en shortlist": "Avanzar condicionado a validación regulatoria / capacidades",
    "Monitorear": "Mantener en monitoreo por sensibilidad, fit o falta de tesis dominante",
    "No priorizar en horizonte actual": "No priorizar en horizonte actual, sin descartarlo permanentemente",
})

recommendation["justificación_ejecutiva"] = recommendation.apply(
    lambda r: (
        f"Score RADAR {r['radar_score']:.3f}, banda {r['banda_atractividad']}, "
        f"presencia Top 5 en {r['n_lines_top5']} líneas y Top 10 en {r['n_lines_top10']} líneas. "
        f"Debe evaluarse con foco en: {r['lines_top5'] if r['lines_top5'] else r['lines_top10']}."
    ), axis=1
)

rec_cols = [
    "country_iso3", "rank", "radar_score", "banda_atractividad", "score_gap_next", "gap_interpretation",
    "lines_top5", "lines_top10", "clasificación", "próxima_fase_SIM", "justificación_ejecutiva",
]

display(style_table(
    recommendation[rec_cols],
    gradient_cols=["radar_score", "score_gap_next"],
    gradient_cmap="RdYlGn",
    format_dict={"radar_score": "{:.3f}", "score_gap_next": "{:.3f}"},
).set_caption("Matriz ejecutiva de recomendación país-línea"))

fig = px.histogram(
    recommendation,
    x="clasificación",
    color="clasificación",
    title="Distribución de recomendaciones ejecutivas por país",
)
fig.update_layout(xaxis_title="Clasificación", yaxis_title="Número de países", height=420)
fig.show()

insight_box(
    "Lectura de recomendación",
    "Los países marcados para avanzar no son recomendaciones de entrada; son candidatos para Fase 4. Los países en monitoreo o no priorizados pueden revaluarse cuando cambien datos, capacidades, objetivo estratégico o horizonte de decisión."
)


,country_iso3,rank,radar_score,banda_atractividad,score_gap_next,gap_interpretation,lines_top5,lines_top10,clasificación,próxima_fase_SIM,justificación_ejecutiva
0,PAN,1,0.691,Alta,0.038,Diferencia material,,AD,No priorizar en horizonte actual,"No priorizar en horizonte actual, sin descartarlo permanentemente","Score RADAR 0.691, banda Alta, presencia Top 5 en 0 líneas y Top 10 en 1 líneas. Debe evaluarse con foco en: AD."
1,CRI,2,0.653,Alta,0.026,Diferencia moderada,,"AD, BD, PF",Mantener en shortlist,Avanzar condicionado a validación regulatoria / capacidades,"Score RADAR 0.653, banda Alta, presencia Top 5 en 0 líneas y Top 10 en 3 líneas. Debe evaluarse con foco en: AD, BD, PF."
2,ESP,3,0.627,Alta,0.007,Diferencia débil,"AD, BD, CIB, IB, PF","AD, BD, CIB, IB, PF",Avanzar a profundización,Avanzar a Fase 4 — Inteligencia cualitativa,"Score RADAR 0.627, banda Alta, presencia Top 5 en 5 líneas y Top 10 en 5 líneas. Debe evaluarse con foco en: AD, BD, CIB, IB, PF."
3,DOM,4,0.620,Media-alta,0.051,Diferencia material,,CIB,Monitorear,"Mantener en monitoreo por sensibilidad, fit o falta de tesis dominante","Score RADAR 0.620, banda Media-alta, presencia Top 5 en 0 líneas y Top 10 en 1 líneas. Debe evaluarse con foco en: CIB."
4,CHL,5,0.569,Media-alta,0.017,Diferencia moderada,"AD, BD, CIB, IB, PF","AD, BD, CIB, IB, PF",Mantener en shortlist,Avanzar condicionado a validación regulatoria / capacidades,"Score RADAR 0.569, banda Media-alta, presencia Top 5 en 5 líneas y Top 10 en 5 líneas. Debe evaluarse con foco en: AD, BD, CIB, IB, PF."
5,USA,6,0.552,Media-alta,0.013,Diferencia débil,"AD, BD, CIB, IB, PF","AD, BD, CIB, IB, PF",Mantener en shortlist,Avanzar condicionado a validación regulatoria / capacidades,"Score RADAR 0.552, banda Media-alta, presencia Top 5 en 5 líneas y Top 10 en 5 líneas. Debe evaluarse con foco en: AD, BD, CIB, IB, PF."
6,URY,7,0.539,Media,0.001,Empate práctico,"AD, PF","AD, BD, CIB, PF",Monitorear,"Mantener en monitoreo por sensibilidad, fit o falta de tesis dominante","Score RADAR 0.539, banda Media, presencia Top 5 en 2 líneas y Top 10 en 4 líneas. Debe evaluarse con foco en: AD, PF."
7,ECU,8,0.538,Media,0.005,Diferencia débil,,,Monitorear,"Mantener en monitoreo por sensibilidad, fit o falta de tesis dominante","Score RADAR 0.538, banda Media, presencia Top 5 en 0 líneas y Top 10 en 0 líneas. Debe evaluarse con foco en: ."
8,MEX,9,0.533,Media,0.003,Empate práctico,,BD,Monitorear,"Mantener en monitoreo por sensibilidad, fit o falta de tesis dominante","Score RADAR 0.533, banda Media, presencia Top 5 en 0 líneas y Top 10 en 1 líneas. Debe evaluarse con foco en: BD."
9,GTM,10,0.530,Baja,0.009,Diferencia débil,,,No priorizar en horizonte actual,"No priorizar en horizonte actual, sin descartarlo permanentemente","Score RADAR 0.530, banda Baja, presencia Top 5 en 0 líneas y Top 10 en 0 líneas. Debe evaluarse con foco en: ."



## 12. Implicaciones para Comité Directivo

El Comité debe decidir:

1. Qué países pasan a profundización cualitativa.
2. Qué líneas de negocio se evalúan en cada país.
3. Qué restricciones deben validarse primero.
4. Qué mercados quedan en monitoreo.
5. Qué criterios o pesos requieren revisión directiva.
6. Qué información adicional se requiere antes de cualquier decisión GO / NO-GO.

**Recomendación de gobierno:** aprobar una shortlist inicial para Fase 4 y solicitar que cada país priorizado tenga una ficha ejecutiva con: tesis de negocio, drivers, restricciones, riesgos regulatorios, capacidades requeridas y modo de entrada preliminar.



## 13. Preguntas anticipadas del Comité y respuestas sugeridas

### 1. ¿Por qué un país cercano puede superar a uno estructuralmente más fuerte?
Porque RADAR no mide solo fortaleza estructural. También incorpora ejecutabilidad desde Colombia y momentum. Un país estructuralmente fuerte puede ser menos prioritario si su distancia operativa, cultural o estratégica aumenta el costo de entrada.

### 2. ¿RADAR recomienda entrar a estos países?
No. RADAR identifica dónde tiene más sentido profundizar. La decisión de entrada exige inteligencia cualitativa, evaluación de capacidades, modo de entrada, due diligence y decisión GO / NO-GO.

### 3. ¿Qué significa que un país sea Top 5 con probabilidad alta?
Significa que bajo múltiples escenarios de pesos razonables el país mantiene posición prioritaria. Es estabilidad de ranking, no probabilidad de éxito comercial.

### 4. ¿Qué hacemos con países excluidos por datos?
No se descartan permanentemente. Se clasifican como no comparables o no candidatos bajo la información disponible y el horizonte actual. Pueden revaluarse si mejora la cobertura de datos o cambia el objetivo estratégico.

### 5. ¿Por qué algunas líneas tienen rankings parecidos?
Porque ciertos países concentran atributos transversales: institucionalidad, digitalización, profundidad financiera y estabilidad. Eso no implica redundancia si las tesis y drivers por línea son distintos.

### 6. ¿El modelo es robusto?
La evidencia disponible indica alta estabilidad determinística y consistencia metodológica. La robustez decisional final debe validarse con las probabilidades Monte Carlo RADAR exactas por país-línea.

### 7. ¿Qué riesgos no captura RADAR?
No captura completamente dinámica competitiva, restricciones regulatorias específicas, apetito de adquisición, disponibilidad de socios, reputación local, capacidad interna ni economía detallada del modo de entrada.

### 8. ¿Qué falta para una decisión de inversión?
Faltan Fase 4 a Fase 8: inteligencia cualitativa, evaluación de capacidades internas, priorización ajustada, modo de entrada y due diligence.

### 9. ¿Cómo se actualiza el modelo?
Debe actualizarse anualmente o ante eventos disruptivos, incorporando nueva información, cambios regulatorios, ajustes de pesos del Comité y evolución de capacidades internas.

### 10. ¿Cómo se conectan estos resultados con capacidades internas de Cibest?
RADAR identifica oportunidad externa. La evaluación de capacidades determina si Cibest puede capturarla. La prioridad final surge de cruzar atractivo externo con capacidad interna.



## 14. Limitaciones y uso responsable

- RADAR no sustituye due diligence regulatorio, financiero, legal ni competitivo.
- TOPSIS es relativo al universo evaluado; si cambia el universo, pueden cambiar los ideales y los scores.
- Las probabilidades Monte Carlo son estabilidad de ranking, no probabilidad de éxito comercial.
- Los pesos expresan una tesis estratégica; no son verdades universales.
- La calidad y cobertura de datos condiciona el resultado.
- Países no priorizados pueden revaluarse en futuros ciclos si cambian datos, estrategia, capacidades o condiciones de mercado.
- Una diferencia ordinal sin gap material no debe convertirse en narrativa ejecutiva de superioridad.



## 15. Cierre ejecutivo

- **Mejores países globales RADAR:** Panamá, Costa Rica, España, República Dominicana y Chile lideran la priorización inicial.
- **Países transversalmente atractivos:** Estados Unidos, Canadá, España y Chile aparecen de forma recurrente en varias líneas de negocio.
- **Mejores países por línea:** IB: USA/ESP/CAN; PF: ESP/CAN/USA; AD: USA/CAN/ESP; BD: USA/ESP/CAN; CIB: CAN/USA/ESP.
- **Países cercanos con tesis selectiva:** Panamá, República Dominicana y Ecuador requieren lectura de proximidad y validación específica por línea.
- **Países estructuralmente fuertes pero lejanos:** Estados Unidos, Canadá, España y Chile deben evaluarse con foco en capacidades, modo de entrada y costo de ejecución.
- **Decisión recomendada:** avanzar a Fase 4 con una shortlist, no tomar decisiones de entrada.
- **Riesgo de interpretación:** no confundir ranking con recomendación de inversión.
- **Siguiente paso:** construir fichas ejecutivas por país-línea priorizado, integrando inteligencia cualitativa, restricciones regulatorias y capacidades internas.

## Síntesis Ejecutiva

RADAR permite pasar de una discusión intuitiva sobre países a una conversación gobernada por evidencia: qué mercados muestran señales atractivas, para qué líneas, con qué drivers, con qué restricciones y con qué robustez. La recomendación para Comité es aprobar la transición de una shortlist priorizada hacia la Fase 4 del proceso SIM, manteniendo explícito que ningún país queda descartado permanentemente y que ninguna decisión de entrada debe tomarse sin las fases posteriores de validación.



## Checklist final de calidad

- [x] El notebook inicia definiendo atractivo.
- [x] RADAR se presenta como fase 3 del proceso SIM, no como decisión final.
- [x] Se explica el embudo completo.
- [x] Se aclara que un mercado no priorizado no está descartado permanentemente.
- [x] Se usan datos reales del master y resultados reales reportados por Notebooks 03/04.
- [x] Se muestran resultados globales y por línea.
- [x] Se explican variables que impulsan atractivo.
- [x] Se explican restricciones críticas.
- [x] Se deja estructura para probabilidades Top-N y recálculo Monte Carlo.
- [x] Se interpreta robustez Monte Carlo y sus límites.
- [x] Se diferencia ranking base vs ranking robusto.
- [x] Se usan bandas y gaps.
- [x] Se formulan recomendaciones para Comité.
- [x] Se anticipan preguntas del Comité.
- [x] Se incluyen limitaciones.
- [x] Cada gráfica tiene interpretación.
- [x] Cada tabla tiene lectura ejecutiva.
